# 08 — Geopolitical Scenario Modeling for Critical Mineral Supply Disruptions

Critical mineral supply chains are concentrated in a small number of countries, creating significant geopolitical risk.  
This notebook models the downstream impact of four key disruption scenarios and identifies alternative suppliers that could offset losses.

## Scenarios modeled

| Scenario | Commodity | Disrupted country | Reduction |
|----------|-----------|------------------|-----------|
| 1 | Rare earth elements | China | −50% |
| 2 | Cobalt | DRC (Congo) | −30% |
| 3 | Nickel | Russia | −100% |
| 4 | Lithium | Australia + Chile | −40% each |

### Methodology

1. **Baseline** — the most recent year of production data is used as the reference state for each commodity.
2. **Disruption** — the production of the specified country (or countries) is reduced by the given fraction.  
3. **Ramp-up** — non-disrupted countries can collectively increase output by `ramp_up_factor × their current production`, distributed proportionally to their baseline shares, and capped at total production lost.
4. **Deficit** — any remaining gap after ramp-up is the net market deficit.

Trade data (imports/exports) provide context on which countries rely on global supply vs. domestic production.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pathlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ---------------------------------------------------------------------------
# DATA_DIR — adjust this path if running from a different working directory
# ---------------------------------------------------------------------------
DATA_DIR = pathlib.Path('../data/bgs_data')

PRODUCTION_FILE = DATA_DIR / 'bgs_critical_minerals_production.csv'
TRADE_FILE      = DATA_DIR / 'bgs_critical_minerals_trade.csv'

print('Production file exists:', PRODUCTION_FILE.exists())
print('Trade file exists:     ', TRADE_FILE.exists())

## 2. Load and clean data

In [ ]:
# ── Production ──────────────────────────────────────────────────────────────
prod_raw = pd.read_csv(PRODUCTION_FILE)

# Normalise text columns
for col in ['commodity', 'statistic_type', 'country']:
    prod_raw[col] = prod_raw[col].astype(str).str.strip()

prod_raw['commodity_lower'] = prod_raw['commodity'].str.lower()
prod_raw['year']     = pd.to_numeric(prod_raw['year'],     errors='coerce')
prod_raw['quantity'] = pd.to_numeric(prod_raw['quantity'], errors='coerce')
prod_raw = prod_raw.dropna(subset=['year', 'quantity'])
prod_raw['year'] = prod_raw['year'].astype(int)

# Keep only Production rows
production = prod_raw[prod_raw['statistic_type'].str.lower() == 'production'].copy()

print(f'Production rows: {len(production):,}')
print(f'Year range: {production["year"].min()} – {production["year"].max()}')
print(f'Commodities: {production["commodity"].nunique()}')

# ── Trade ────────────────────────────────────────────────────────────────────
trade_raw = pd.read_csv(TRADE_FILE)

for col in ['commodity', 'statistic_type', 'country']:
    trade_raw[col] = trade_raw[col].astype(str).str.strip()

trade_raw['commodity_lower'] = trade_raw['commodity'].str.lower()
trade_raw['year']     = pd.to_numeric(trade_raw['year'],     errors='coerce')
trade_raw['quantity'] = pd.to_numeric(trade_raw['quantity'], errors='coerce')
trade_raw = trade_raw.dropna(subset=['year', 'quantity'])
trade_raw['year'] = trade_raw['year'].astype(int)

trade = trade_raw[trade_raw['statistic_type'].str.lower().isin(['exports', 'imports'])].copy()

print(f'\nTrade rows: {len(trade):,}')
print(f'Trade stat types: {trade["statistic_type"].unique()}')

### 2.1 Commodity lookup helper

Because commodity names vary slightly across rows, we use a case-insensitive substring match so callers can pass plain names like `'cobalt'` or `'rare earth'`.

In [ ]:
def _filter_commodity(df: pd.DataFrame, commodity: str) -> pd.DataFrame:
    """Return rows whose commodity_lower contains the search term."""
    term = commodity.lower().strip()
    return df[df['commodity_lower'].str.contains(term, na=False, regex=False)]


# Spot-check commodity coverage for the four scenarios
scenarios = [
    ('rare earth', 'China'),
    ('cobalt',     'Congo'),
    ('nickel',     'Russia'),
    ('lithium',    'Australia'),
]

print('Commodity match check (production):')
for term, country in scenarios:
    sub = _filter_commodity(production, term)
    latest = sub['year'].max() if len(sub) else 'n/a'
    commodities = sub['commodity'].unique()[:3].tolist() if len(sub) else []
    print(f'  {term!r:20s} → {len(sub):5,} rows, latest year={latest}, names={commodities}')

## 3. Baseline function

`get_baseline(commodity)` returns a snapshot of the most recent year of available data for the matching commodity.  
It aggregates across sub-commodities so each country appears once.

In [ ]:
def get_baseline(commodity: str) -> dict:
    """
    Build a baseline snapshot for *commodity* using the most recent year
    of production data that has at least 3 reporting countries.

    Returns
    -------
    dict with keys:
        commodity        : matched commodity label (most common)
        year             : baseline year
        production       : {country: quantity}
        exports          : {country: quantity}   (may be empty)
        imports          : {country: quantity}   (may be empty)
        total_production : float
        units            : str
    """
    # --- Production baseline ------------------------------------------------
    sub = _filter_commodity(production, commodity)
    if sub.empty:
        raise ValueError(f'No production data found for commodity matching "{commodity}"')

    # Walk backwards from most recent year until we have >=3 countries
    for yr in sorted(sub['year'].unique(), reverse=True):
        yr_data = sub[sub['year'] == yr]
        if yr_data['country'].nunique() >= 3:
            latest_year = yr
            break
    else:
        latest_year = sub['year'].max()
        yr_data = sub[sub['year'] == latest_year]

    prod_by_country = (
        yr_data
        .groupby('country')['quantity']
        .sum()
        .sort_values(ascending=False)
        .to_dict()
    )

    units = yr_data['units'].mode()[0] if not yr_data['units'].empty else 'tonnes'
    commodity_label = yr_data['commodity'].mode()[0]

    # --- Trade baseline (best available year <= latest_year) ----------------
    trade_sub = _filter_commodity(trade, commodity)

    def _trade_dict(stat_type: str) -> dict:
        t = trade_sub[trade_sub['statistic_type'].str.lower() == stat_type]
        if t.empty:
            return {}
        # Use the most recent year <= baseline production year
        available = t[t['year'] <= latest_year]['year']
        if available.empty:
            return {}
        best_yr = available.max()
        return (
            t[t['year'] == best_yr]
            .groupby('country')['quantity']
            .sum()
            .sort_values(ascending=False)
            .to_dict()
        )

    exports_dict = _trade_dict('exports')
    imports_dict = _trade_dict('imports')

    total_prod = sum(prod_by_country.values())

    return {
        'commodity':        commodity_label,
        'year':             latest_year,
        'production':       prod_by_country,
        'exports':          exports_dict,
        'imports':          imports_dict,
        'total_production': total_prod,
        'units':            units,
    }


# --- Quick sanity check -----------------------------------------------------
for term in ['rare earth', 'cobalt', 'nickel', 'lithium']:
    b = get_baseline(term)
    top3 = list(b['production'].items())[:3]
    print(f'{b["commodity"][:45]:45s}  year={b["year"]}  total={b["total_production"]:,.0f} {b["units"]}  top3={top3}')

## 4. Simulation engine

`simulate_disruption` applies one or more country-level production shocks and models the compensating ramp-up from remaining producers.

In [ ]:
def simulate_disruption(
    commodity: str,
    disruptions: dict,          # {country_name_fragment: fraction_to_reduce}
    ramp_up_factor: float = 0.5,
) -> tuple[pd.DataFrame, dict]:
    """
    Simulate supply disruptions for *commodity*.

    Parameters
    ----------
    commodity      : substring to match commodity name
    disruptions    : mapping of country name fragment → fraction of production
                     to remove (0 < fraction ≤ 1). Country matching is
                     case-insensitive substring.
    ramp_up_factor : fraction by which non-disrupted producers can collectively
                     increase their output to offset lost supply.

    Returns
    -------
    (results_df, summary_dict)

    results_df columns:
        country, baseline_production, is_disrupted, production_lost,
        scenario_production, ramp_up_added, final_production

    summary_dict keys:
        commodity, year, total_before, total_lost, total_ramp_up,
        total_after, net_deficit, deficit_pct, units
    """
    baseline = get_baseline(commodity)
    prod = baseline['production']
    units = baseline['units']
    year  = baseline['year']

    rows = []
    total_before = sum(prod.values())
    total_lost = 0.0

    # ------------------------------------------------------------------
    # Step 1: apply disruptions
    # ------------------------------------------------------------------
    disrupted_countries = set()
    matched_disruptions = {}   # exact country → fraction

    for country, qty in prod.items():
        lost = 0.0
        for key, frac in disruptions.items():
            if key.lower() in country.lower():
                lost = qty * min(max(frac, 0.0), 1.0)
                disrupted_countries.add(country)
                break
        scenario_prod = qty - lost
        total_lost += lost
        rows.append({
            'country':              country,
            'baseline_production':  qty,
            'is_disrupted':         lost > 0,
            'production_lost':      lost,
            'scenario_production':  scenario_prod,
            'ramp_up_added':        0.0,
            'final_production':     scenario_prod,
        })

    # ------------------------------------------------------------------
    # Step 2: distribute ramp-up among non-disrupted countries
    # ------------------------------------------------------------------
    non_disrupted_total = sum(
        r['scenario_production'] for r in rows if not r['is_disrupted']
    )
    max_ramp_up = non_disrupted_total * ramp_up_factor
    actual_ramp_up = min(max_ramp_up, total_lost)  # cannot exceed lost supply

    if non_disrupted_total > 0:
        for r in rows:
            if not r['is_disrupted']:
                share = r['scenario_production'] / non_disrupted_total
                added = actual_ramp_up * share
                r['ramp_up_added']    = added
                r['final_production'] = r['scenario_production'] + added

    # ------------------------------------------------------------------
    # Step 3: assemble outputs
    # ------------------------------------------------------------------
    results_df = pd.DataFrame(rows).sort_values('baseline_production', ascending=False)

    total_after  = results_df['final_production'].sum()
    net_deficit  = total_before - total_after
    deficit_pct  = (net_deficit / total_before * 100) if total_before > 0 else 0.0

    summary = {
        'commodity':    baseline['commodity'],
        'year':         year,
        'total_before': total_before,
        'total_lost':   total_lost,
        'total_ramp_up': actual_ramp_up,
        'total_after':  total_after,
        'net_deficit':  net_deficit,
        'deficit_pct':  deficit_pct,
        'units':        units,
    }

    return results_df, summary


# ── Quick test: China −50% rare earths ──────────────────────────────────────
df_test, s_test = simulate_disruption('rare earth', {'China': 0.50})
print(f"Rare earths — China −50%")
print(f"  Before:    {s_test['total_before']:>12,.0f} {s_test['units']}")
print(f"  Lost:      {s_test['total_lost']:>12,.0f}")
print(f"  Ramp-up:   {s_test['total_ramp_up']:>12,.0f}")
print(f"  After:     {s_test['total_after']:>12,.0f}")
print(f"  Deficit:   {s_test['net_deficit']:>12,.0f}  ({s_test['deficit_pct']:.1f}%)")

## 5. Four-scenario comparison

Each panel shows the top 8 producers for the commodity, with grouped bars comparing **baseline** versus **post-scenario** production.  
Disrupted countries are highlighted; the grey bars show ramp-up contribution from other producers.

In [ ]:
SCENARIOS = [
    # (title,           commodity,    disruptions dict,                    ramp_up)
    ('Rare Earths\nChina −50%',   'rare earth',  {'China': 0.50},                        0.5),
    ('Cobalt\nDRC −30%',          'cobalt',      {'Congo': 0.30},                        0.5),
    ('Nickel\nRussia −100%',      'nickel',      {'Russia': 1.00},                       0.5),
    ('Lithium\nAustralia+Chile −40%', 'lithium', {'Australia': 0.40, 'Chile': 0.40},     0.5),
]

TOP_N = 8   # producers to display per subplot

results_store = {}   # store for later use
summaries     = {}

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[s[0].replace('\n', ' — ') for s in SCENARIOS],
    vertical_spacing=0.22,
    horizontal_spacing=0.12,
)

PALETTE = {
    'baseline':    '#4C72B0',
    'scenario':    '#55A868',
    'disrupted':   '#C44E52',
    'ramp_up':     '#8172B2',
}

for idx, (title, commodity, disruptions, ramp_up) in enumerate(SCENARIOS):
    row = idx // 2 + 1
    col = idx %  2 + 1

    res_df, summary = simulate_disruption(commodity, disruptions, ramp_up)
    results_store[commodity] = res_df
    summaries[commodity]     = summary

    # Top-N by baseline production
    top = res_df.head(TOP_N).copy()

    countries  = top['country'].tolist()
    baseline_v = top['baseline_production'].tolist()
    scenario_v = top['final_production'].tolist()
    colors_scenario = [
        PALETTE['disrupted'] if d else PALETTE['scenario']
        for d in top['is_disrupted']
    ]

    show_legend = (idx == 0)

    fig.add_trace(
        go.Bar(
            name='Baseline',
            x=countries,
            y=baseline_v,
            marker_color=PALETTE['baseline'],
            opacity=0.85,
            legendgroup='baseline',
            showlegend=show_legend,
        ),
        row=row, col=col,
    )
    fig.add_trace(
        go.Bar(
            name='Scenario (disrupted)',
            x=[c for c, d in zip(countries, top['is_disrupted']) if d],
            y=[v for v, d in zip(scenario_v, top['is_disrupted']) if d],
            marker_color=PALETTE['disrupted'],
            legendgroup='disrupted',
            showlegend=show_legend,
        ),
        row=row, col=col,
    )
    fig.add_trace(
        go.Bar(
            name='Scenario (ramp-up)',
            x=[c for c, d in zip(countries, top['is_disrupted']) if not d],
            y=[v for v, d in zip(scenario_v, top['is_disrupted']) if not d],
            marker_color=PALETTE['scenario'],
            legendgroup='ramp_up',
            showlegend=show_legend,
        ),
        row=row, col=col,
    )

    # Annotate deficit in subplot title area
    fig.add_annotation(
        text=f"Deficit: {summary['deficit_pct']:.1f}% of global supply",
        xref=f'x{idx+1} domain' if idx > 0 else 'x domain',
        yref=f'y{idx+1} domain' if idx > 0 else 'y domain',
        x=0.5, y=1.08,
        showarrow=False,
        font=dict(size=11, color='#555'),
        xanchor='center',
        row=row, col=col,
    )

fig.update_layout(
    title=dict(
        text='Geopolitical Disruption Scenarios — Baseline vs Post-Disruption Production (Top 8 Producers)',
        font=dict(size=16),
        x=0.5,
    ),
    barmode='group',
    height=780,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig.update_xaxes(tickangle=-35)
fig.update_yaxes(gridcolor='#eee')

fig.show()

# Print summary table
print(f"{'Scenario':<30} {'Before':>14} {'Lost':>14} {'Ramp-up':>14} {'After':>14} {'Deficit %':>10}")
print('-' * 88)
for commodity, s in summaries.items():
    label = f"{s['commodity'][:28]}"
    print(f"{label:<30} {s['total_before']:>14,.0f} {s['total_lost']:>14,.0f} "
          f"{s['total_ramp_up']:>14,.0f} {s['total_after']:>14,.0f} "
          f"{s['deficit_pct']:>9.1f}%")

## 6. Alternative supplier identification

When a major supplier is disrupted, which other countries have the capacity to fill the gap?

`find_alternative_suppliers` estimates two dimensions of spare capacity:

- **Spare capacity** — difference between a country's peak production in the lookback window and its most recent production.  
  This represents idled capacity that could potentially be reactivated quickly.
- **Growth trend** — linear regression slope (tonnes/year) over the lookback window.  
  Countries with rising trends are more likely to sustain increased output.
- **Ramp-up potential** — composite score = `spare_capacity + max(0, trend × 3)` (equivalent to three additional years of trend growth).

In [ ]:
from scipy import stats as scipy_stats


def find_alternative_suppliers(
    commodity: str,
    disrupted_country: str,
    lookback_years: int = 10,
    top_n: int = 10,
) -> pd.DataFrame:
    """
    Identify alternative suppliers for *commodity* if *disrupted_country*
    is taken off-line.

    Parameters
    ----------
    commodity         : commodity substring to match
    disrupted_country : country name fragment to exclude
    lookback_years    : historical window for spare-capacity & trend analysis
    top_n             : number of alternatives to return

    Returns
    -------
    DataFrame sorted by ramp_up_potential, with columns:
        country, current_production, peak_production, spare_capacity,
        trend_per_year, ramp_up_potential, production_share_pct
    """
    baseline  = get_baseline(commodity)
    latest_yr = baseline['year']
    cutoff_yr = latest_yr - lookback_years

    sub = _filter_commodity(production, commodity)
    sub = sub[sub['year'] >= cutoff_yr]

    # Exclude the disrupted country
    sub = sub[~sub['country'].str.lower().str.contains(disrupted_country.lower(), regex=False)]

    # Aggregate quantity per country per year (handles multi-row sub-commodities)
    annual = sub.groupby(['country', 'year'])['quantity'].sum().reset_index()

    records = []
    for country, grp in annual.groupby('country'):
        grp = grp.sort_values('year')

        current = grp[grp['year'] == latest_yr]['quantity'].sum()
        if current == 0:
            # Use last available year if not present in baseline year
            current = grp['quantity'].iloc[-1] if len(grp) else 0.0

        peak    = grp['quantity'].max()
        spare   = max(0.0, peak - current)

        # Trend via linear regression (require >=3 data points)
        trend = 0.0
        if len(grp) >= 3:
            slope, *_ = scipy_stats.linregress(grp['year'], grp['quantity'])
            trend = slope

        ramp_potential = spare + max(0.0, trend * 3)

        records.append({
            'country':            country,
            'current_production': current,
            'peak_production':    peak,
            'spare_capacity':     spare,
            'trend_per_year':     trend,
            'ramp_up_potential':  ramp_potential,
        })

    df = pd.DataFrame(records)
    if df.empty:
        return df

    total_current = df['current_production'].sum()
    df['production_share_pct'] = (
        df['current_production'] / total_current * 100
        if total_current > 0 else 0.0
    )

    df = df.sort_values('ramp_up_potential', ascending=False).head(top_n)
    return df.reset_index(drop=True)


# ── Apply: rare earths with China disrupted ──────────────────────────────────
alt_ree = find_alternative_suppliers('rare earth', 'China', lookback_years=10, top_n=10)
print('Top alternative rare-earth suppliers if China is disrupted:')
display_cols = ['country', 'current_production', 'spare_capacity', 'trend_per_year', 'ramp_up_potential']
print(alt_ree[display_cols].to_string(index=False, float_format='{:,.0f}'.format))

### 6.1 Stacked bar — current production + spare capacity

In [ ]:
def plot_alternative_suppliers(
    alt_df: pd.DataFrame,
    commodity: str,
    disrupted_country: str,
    units: str = 'tonnes',
) -> go.Figure:
    """Stacked bar: current_production (base) + spare_capacity (top)."""

    df = alt_df.sort_values('ramp_up_potential', ascending=True)  # ascending for horizontal

    fig = go.Figure()

    fig.add_trace(go.Bar(
        name='Current production',
        y=df['country'],
        x=df['current_production'],
        orientation='h',
        marker_color='#4C72B0',
        hovertemplate='%{y}<br>Current: %{x:,.0f}<extra></extra>',
    ))

    fig.add_trace(go.Bar(
        name='Spare capacity (peak − current)',
        y=df['country'],
        x=df['spare_capacity'],
        orientation='h',
        marker_color='#DD8452',
        hovertemplate='%{y}<br>Spare: %{x:,.0f}<extra></extra>',
    ))

    # Mark ramp-up potential with a scatter point
    total_potential = df['current_production'] + df['spare_capacity']
    fig.add_trace(go.Scatter(
        name='Ramp-up potential',
        y=df['country'],
        x=df['ramp_up_potential'],
        mode='markers',
        marker=dict(symbol='diamond', size=10, color='#C44E52', line=dict(width=1, color='white')),
        hovertemplate='%{y}<br>Ramp-up potential: %{x:,.0f}<extra></extra>',
    ))

    fig.update_layout(
        title=dict(
            text=(
                f'Alternative Suppliers for {commodity.title()}<br>'
                f'<sup>If {disrupted_country} is disrupted — ranked by ramp-up potential</sup>'
            ),
            font=dict(size=15),
            x=0.5,
        ),
        barmode='stack',
        xaxis_title=f'Production ({units})',
        yaxis_title='Country',
        height=480,
        legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(gridcolor='#eee'),
    )
    return fig


baseline_ree = get_baseline('rare earth')
fig_alt = plot_alternative_suppliers(
    alt_ree,
    commodity='Rare Earth Elements',
    disrupted_country='China',
    units=baseline_ree['units'],
)
fig_alt.show()

### 6.2 Alternative suppliers for all four scenarios

In [ ]:
ALT_SCENARIOS = [
    ('rare earth', 'China',     'Rare Earth Elements'),
    ('cobalt',     'Congo',     'Cobalt'),
    ('nickel',     'Russia',    'Nickel'),
    ('lithium',    'Australia', 'Lithium (ex-Australia)'),
]

for commodity, disrupted, label in ALT_SCENARIOS:
    alt = find_alternative_suppliers(commodity, disrupted, lookback_years=10, top_n=8)
    b   = get_baseline(commodity)
    if alt.empty:
        print(f'No alternatives found for {label}')
        continue
    fig = plot_alternative_suppliers(alt, label, disrupted, units=b['units'])
    fig.show()

## 7. Deficit severity heatmap

Sweep across ramp-up factors (0 – 100%) and disruption severities (10 – 100%) for each scenario to understand how the deficit evolves under different assumptions.

In [ ]:
def deficit_heatmap(
    commodity: str,
    disrupted_countries: list[str],
    title: str,
    ramp_up_range=None,
    disruption_range=None,
) -> go.Figure:
    """Plot a 2-D sensitivity heatmap of deficit_pct."""
    if ramp_up_range is None:
        ramp_up_range = np.arange(0.0, 1.05, 0.1)
    if disruption_range is None:
        disruption_range = np.arange(0.1, 1.05, 0.1)

    z = np.zeros((len(ramp_up_range), len(disruption_range)))

    for i, ruf in enumerate(ramp_up_range):
        for j, df_frac in enumerate(disruption_range):
            disruptions = {c: df_frac for c in disrupted_countries}
            try:
                _, s = simulate_disruption(commodity, disruptions, ramp_up_factor=ruf)
                z[i, j] = s['deficit_pct']
            except Exception:
                z[i, j] = np.nan

    x_labels = [f'{int(v*100)}%' for v in disruption_range]
    y_labels = [f'{int(v*100)}%' for v in ramp_up_range]

    fig = go.Figure(go.Heatmap(
        z=z,
        x=x_labels,
        y=y_labels,
        colorscale='RdYlGn_r',
        zmin=0,
        zmax=60,
        colorbar=dict(title='Deficit %'),
        hovertemplate=(
            'Disruption: %{x}<br>Ramp-up capacity: %{y}<br>'
            'Deficit: %{z:.1f}%<extra></extra>'
        ),
    ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=14), x=0.5),
        xaxis_title='Disruption severity (% reduction)',
        yaxis_title='Ramp-up capacity of other producers',
        height=420,
        plot_bgcolor='white',
        paper_bgcolor='white',
    )
    return fig


heatmap_configs = [
    ('rare earth', ['China'],              'Rare Earths — China Disruption Sensitivity'),
    ('cobalt',     ['Congo'],              'Cobalt — DRC Disruption Sensitivity'),
    ('nickel',     ['Russia'],             'Nickel — Russia Disruption Sensitivity'),
    ('lithium',    ['Australia', 'Chile'], 'Lithium — Australia + Chile Disruption Sensitivity'),
]

for commodity, countries, title in heatmap_configs:
    fig_hm = deficit_heatmap(commodity, countries, title)
    fig_hm.show()

## 8. Custom scenario builder

Modify the variables in the cell below to model any commodity and disruption combination.

**How to use:**
1. Change `COMMODITY` to any mineral name (e.g. `'graphite'`, `'manganese'`, `'titanium'`).
2. Add or edit entries in `DISRUPTIONS` — keys are case-insensitive country name fragments,  
   values are the fraction of production to remove (0.0 – 1.0).
3. Adjust `RAMP_UP_FACTOR` (0 = no ramp-up, 1 = all other producers can double output to absorb loss).
4. Run the cell.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║              C U S T O M   S C E N A R I O   B U I L D E R          ║
# ╚══════════════════════════════════════════════════════════════════════╝

COMMODITY      = 'cobalt'            # commodity to analyse (substring match)
DISRUPTIONS    = {
    'Congo':  0.50,                  # DRC −50%
    'Zambia': 0.25,                  # Zambia −25%
}
RAMP_UP_FACTOR = 0.40                # other producers can raise output by 40%

# ─────────────────────────────────────────────────────────────────────
#  Do not edit below this line — it runs the simulation and plots results
# ─────────────────────────────────────────────────────────────────────

custom_df, custom_summary = simulate_disruption(COMMODITY, DISRUPTIONS, RAMP_UP_FACTOR)

# ── Text summary ────────────────────────────────────────────────────
print(f"Custom scenario: {COMMODITY.title()}")
print(f"  Disruptions: {DISRUPTIONS}")
print(f"  Ramp-up factor: {RAMP_UP_FACTOR:.0%}")
print()
print(f"  Baseline total production : {custom_summary['total_before']:>14,.0f} {custom_summary['units']}")
print(f"  Production lost            : {custom_summary['total_lost']:>14,.0f}")
print(f"  Offset by ramp-up          : {custom_summary['total_ramp_up']:>14,.0f}")
print(f"  Post-disruption total      : {custom_summary['total_after']:>14,.0f}")
print(f"  Net deficit                : {custom_summary['net_deficit']:>14,.0f}  ({custom_summary['deficit_pct']:.1f}%)")

# ── Bar chart ───────────────────────────────────────────────────────
top_custom = custom_df.head(12).copy()

fig_custom = go.Figure()

fig_custom.add_trace(go.Bar(
    name='Baseline production',
    x=top_custom['country'],
    y=top_custom['baseline_production'],
    marker_color='#4C72B0',
    opacity=0.85,
))

fig_custom.add_trace(go.Bar(
    name='Post-disruption (disrupted)',
    x=top_custom.loc[top_custom['is_disrupted'], 'country'],
    y=top_custom.loc[top_custom['is_disrupted'], 'final_production'],
    marker_color='#C44E52',
))

fig_custom.add_trace(go.Bar(
    name='Post-disruption (with ramp-up)',
    x=top_custom.loc[~top_custom['is_disrupted'], 'country'],
    y=top_custom.loc[~top_custom['is_disrupted'], 'final_production'],
    marker_color='#55A868',
))

disruption_label = ', '.join(f'{k} −{v:.0%}' for k, v in DISRUPTIONS.items())
fig_custom.update_layout(
    title=dict(
        text=(
            f'Custom Scenario: {COMMODITY.title()} — {disruption_label}<br>'
            f'<sup>Deficit: {custom_summary["deficit_pct"]:.1f}% | '
            f'Ramp-up factor: {RAMP_UP_FACTOR:.0%} | '
            f'Baseline year: {custom_summary["year"]}</sup>'
        ),
        font=dict(size=14),
        x=0.5,
    ),
    barmode='group',
    xaxis_title='Country',
    yaxis_title=f'Production ({custom_summary["units"]})',
    height=460,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(tickangle=-35),
    yaxis=dict(gridcolor='#eee'),
)

fig_custom.show()

# ── Country-level breakdown table ───────────────────────────────────
print('\nCountry-level breakdown (top 12 producers):')
display_custom = custom_df[['country', 'baseline_production', 'production_lost',
                             'ramp_up_added', 'final_production', 'is_disrupted']].head(12)
print(display_custom.to_string(index=False, float_format='{:,.0f}'.format))

## 9. Key findings summary

In [ ]:
summary_rows = []
for commodity, s in summaries.items():
    summary_rows.append({
        'Commodity':          s['commodity'],
        'Baseline year':      s['year'],
        'Before (t)':         f"{s['total_before']:,.0f}",
        'Lost (t)':           f"{s['total_lost']:,.0f}",
        'Ramp-up (t)':        f"{s['total_ramp_up']:,.0f}",
        'After (t)':          f"{s['total_after']:,.0f}",
        'Net deficit (t)':    f"{s['net_deficit']:,.0f}",
        'Deficit %':          f"{s['deficit_pct']:.1f}%",
    })

summary_table = pd.DataFrame(summary_rows)
print(summary_table.to_string(index=False))

# ── Deficit bar chart ────────────────────────────────────────────────
fig_sum = go.Figure(go.Bar(
    x=[r['Commodity'][:30] for r in summary_rows],
    y=[s['deficit_pct'] for s in summaries.values()],
    marker_color=['#C44E52' if s['deficit_pct'] > 20 else '#DD8452'
                  for s in summaries.values()],
    text=[f"{s['deficit_pct']:.1f}%" for s in summaries.values()],
    textposition='outside',
    hovertemplate='%{x}<br>Deficit: %{y:.1f}%<extra></extra>',
))

fig_sum.update_layout(
    title=dict(
        text='Net Global Supply Deficit by Scenario (50% ramp-up assumed)',
        font=dict(size=14),
        x=0.5,
    ),
    xaxis_title='Commodity',
    yaxis_title='Deficit (% of baseline supply)',
    yaxis=dict(range=[0, max(s['deficit_pct'] for s in summaries.values()) * 1.25 + 5],
               gridcolor='#eee'),
    height=400,
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False,
)

fig_sum.show()